In [6]:
import subprocess
import pandas as pd

ModuleNotFoundError: No module named 'pandas'

In [4]:
url = "https://api-pre.library.tamu.edu/fcrepo/rest/bb/97/f2/3e/bb97f23e-803a-4bd6-8406-06802623554c/scifi-exhibit-2021%5B2%5D"
result = subprocess.run(["ferry", "ferry", url], capture_output= True, text= True)
print(result.stdout)
print(result.stderr)




100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 1/1 [00:02<00:00,  2.28s/it]



In [4]:
import os
import pandas as pd
import subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.parse import urlparse

INPUT_CSV = "Fedora Collections.csv"
OUTPUT_CSV = "final_output.csv"
URL_COLUMN = "URI"
OUTPUT_FOLDER = "generated_csvs"
MAX_WORKERS = 10
TIMEOUT_SECONDS = 120

# Replace with your actual ferry command
BASE_COMMAND = ["ferry"]

# ==========================
# Setup
# ==========================
os.makedirs(OUTPUT_FOLDER, exist_ok=True)


def extract_filename_from_url(url):
    path = urlparse(url).path
    return path.rstrip("/").split("/")[-1]


def run_ferry(url):
    filename = extract_filename_from_url(url) + ".csv"
    full_output_path = os.path.join(OUTPUT_FOLDER, filename)

    try:
        result = subprocess.run(
            BASE_COMMAND + [url],
            capture_output=True,
            text=True,
            timeout=TIMEOUT_SECONDS
        )

        if result.returncode != 0:
            return {
                "url": url,
                "error": result.stderr.strip() or f"Failed (code {result.returncode})",
                "file": ""
            }

        # Save one CSV per URL
        with open(full_output_path, "w", encoding="utf-8") as f:
            f.write(result.stdout)

        return {
            "url": url,
            "error": "",
            "file": filename  # Only filename, no folder path
        }

    except subprocess.TimeoutExpired:
        return {"url": url, "error": "Timeout", "file": ""}
    except Exception as e:
        return {"url": url, "error": str(e), "file": ""}


def main():
    df = pd.read_csv(INPUT_CSV)

    if URL_COLUMN not in df.columns:
        raise ValueError(f"Column '{URL_COLUMN}' not found in CSV.")

    results = []

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(run_ferry, url) for url in df[URL_COLUMN]]

        for future in as_completed(futures):
            results.append(future.result())

    result_map = {r["url"]: r for r in results}

    df["error"] = df[URL_COLUMN].map(lambda x: result_map[x]["error"])
    df["generated_csv"] = df[URL_COLUMN].map(lambda x: result_map[x]["file"])

    df.to_csv(OUTPUT_CSV, index=False)

    print("Done.")
    print(f"All individual CSVs saved in folder: {OUTPUT_FOLDER}")
    print(f"Final report saved as: {OUTPUT_CSV}")


if __name__ == "__main__":
    main()

Done.
All individual CSVs saved in folder: generated_csvs
Final report saved as: final_output.csv


In [ ]:
import csv
import subprocess
import os

INPUT_FILE = "Fedora Collections.csv"
LOG_FILE = "ferry_log.txt"

def get_slug_from_url(url):
    return url.rstrip("/").split("/")[-1]

with open(INPUT_FILE, newline="", encoding="utf-8") as file, \
     open(LOG_FILE, "a", encoding="utf-8") as log:

    reader = csv.DictReader(file)

    for row in reader:
        url = row["URI"].strip()

        if not url:
            continue

        slug = get_slug_from_url(url)
        expected_csv = f"{slug}.csv"

        if os.path.exists(expected_csv):
            print(f"Skipping {slug} (already exists)")
            continue

        print(f"Processing: {slug}")

        try:
            result = subprocess.run(
                ["ferry", "ferry", url],
                capture_output=True,
                text=True,
                encoding="utf-8",
                errors="replace"
            )

            if result.returncode != 0:
                log.write("=====================================\n")
                log.write(f"Collection: {slug}\n")
                log.write(f"URL: {url}\n")
                log.write("ERROR MESSAGE:\n")
                log.write(result.stderr + "\n")
                log.write("=====================================\n\n")
                print("  -> ERROR")
                continue

            if os.path.exists(expected_csv):
                print("  -> CSV generated successfully")
            else:
                log.write("=====================================\n")
                log.write(f"Collection: {slug}\n")
                log.write(f"URL: {url}\n")
                log.write("CSV file was NOT generated.\n")
                log.write("=====================================\n\n")
                print("  -> CSV NOT FOUND")

        except Exception as e:
            log.write("=====================================\n")
            log.write(f"Collection: {slug}\n")
            log.write(f"URL: {url}\n")
            log.write(f"PYTHON EXCEPTION: {str(e)}\n")
            log.write("=====================================\n\n")
            print("  -> PYTHON ERROR")

Skipping com_class_rosters (already exists)
Skipping charting-texas (already exists)
Skipping Stephen_Powys_Marks_London_Collection (already exists)
Processing: BasbanesEternalPassion
  -> CSV generated successfully
Processing: london-collection
  -> CSV generated successfully
Processing: basbanes-texts
  -> CSV generated successfully
Processing: berger_cloonan_batch_a
  -> CSV generated successfully
Processing: scifi-exhibit-2021
  -> CSV generated successfully
Processing: fb006cbc-d2e5-4fcb-9321-62b6385720d0
  -> CSV generated successfully
Processing: berger_cloonan_batch_3
  -> CSV generated successfully
Processing: berger_cloonan_batch_4
